# Seminar 3. Language modelling

In this seminar, we will try to generate some jokes. [Data source](https://t.me/NeuralShit/2321).

#### Agenda
0. Tokenization
1. Introduction to Language Modeling
    * Probabilistic approach
    * Building and sampling from a statistical n-gram model
2. Perplexity
3. Sequence Modeling with LSTMs

In [76]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from tqdm.auto import tqdm
%matplotlib inline

In [77]:
with open('data/anek.txt', 'r') as f:
    aneki = f.read().strip().replace('<|startoftext|>', '').split('\n\n')

In [78]:
aneki[42]

'Кто сказал, что солдат мечтает стать генералом? Солдат мечтает стать хлеборезом.'

## 0. Tokenization

By now we knew only simple tokenization, simply dividing by spaces/punctuation. Here we introduce BPE (Byte-pair Encoder) Tokenizer:

![BPE Tokenizer](pictures/bpe-example.png)

At some point we want to have the best of two worlds: tokenization by characters (to have a smaller vocab) and tokenization by words (to have more understanding of semantic structure). It starts with a vocabulary of individual characters and iteratively merges the most frequent pairs of symbols (characters or sequences) in the corpus to form new tokens. We continue to "glue" those tokens together until the required vocabulary size is reached.

For example, if the word “lower” appears frequently, BPE might first merge 'l' + 'o' → 'lo', then 'lo' + 'w' → 'low', and later 'low' + 'e' → 'lowe', and finally 'lowe' + 'r' → 'lower'. Rare words like "lowercase" may be split into familiar subwords: ['lower', 'case']. BPE is especially useful for handling rare or unknown words, as it avoids using an <unk> token by breaking them into smaller known subwords.

In [79]:
from bpe import Encoder

In [80]:
# pct_bpe - proportion of tokens that obtained via BPE. Other are the most frequent words.
encoder = Encoder(50000, ngram_max=6, pct_bpe=0.95)
encoder.fit(aneki)

In [81]:
len(encoder.bpe_vocab)

47500

In [82]:
list(encoder.bpe_vocab.items())[:10], list(encoder.bpe_vocab.items())[-10:]

([('__sow', 2500),
  ('__eow', 2501),
  ('о', 2502),
  ('а', 2503),
  ('е', 2504),
  ('и', 2505),
  ('т', 2506),
  ('н', 2507),
  ('р', 2508),
  ('с', 2509)],
 [('ньюто', 49990),
  ('ньютон', 49991),
  ('геи', 49992),
  ('дилы', 49993),
  ('одилы', 49994),
  ('абар', 49995),
  ('заха', 49996),
  ('еревни', 49997),
  ('енятьс', 49998),
  ('изюм', 49999)])

In [83]:
example = aneki[42]
print(encoder.tokenize(example))
print(next(encoder.transform([example])))
print(next(encoder.inverse_transform(encoder.transform([example]))))

['кто', 'сказал', ',', 'что', 'солдат', '__sow', 'мечтае', 'т', '__eow', 'стать', '__sow', 'генера', 'лом', '__eow', '?', 'солдат', '__sow', 'мечтае', 'т', '__eow', 'стать', '__sow', 'хлебо', 'резо', 'м', '__eow', '.']
[59, 172, 2, 9, 1509, 2500, 20745, 2506, 2501, 385, 2500, 15260, 3297, 2501, 23, 1509, 2500, 20745, 2506, 2501, 385, 2500, 25012, 26588, 2441, 2501, 3]
кто сказал , что солдат мечтает стать генералом ? солдат мечтает стать хлеборезом .


At this point we do not really like the formatting, so let's use the more common one: if a token is not the beginning of the word, we add `##` to the front of the token.

So we are going to turn

```['кто', 'сказал', ',', 'что', 'солдат', '__sow', 'мечтае', 'т', '__eow', 'стать', '__sow', 'генера', 'лом', '__eow', '?', 'солдат', '__sow', 'мечтае', 'т', '__eow', 'стать', '__sow', 'хлебо', 'резо', 'м', '__eow', '.']```

into

```['кто', 'сказал', ',', 'что', 'солдат', 'мечтае', '##т', 'стать', 'генера', '##лом', '?', 'солдат', 'мечтае', '##т', 'стать', 'хлебо', '##резо', '##м', '.']```

In [84]:
# def tokenize_bpe(text):
#     tokenized = encoder.tokenize(text)
#     clear_tokenized = []

#     # YOUR CODE
#     # YOUR CODE
#     # YOUR CODE
    
#     return clear_tokenized

In [85]:
def tokenize_bpe(text):
    tokenized = encoder.tokenize(text)
    clear_tokenized = []
    # YOUR CODE
    # SOLUTION BELOW:
    first = False
    saw_eow = True
    for token in tokenized:
        if token == '__sow':
            saw_eow = False
            first = True
            continue
        elif token == '__eow':
            saw_eow = True
            continue
        else:
            if first or saw_eow:
                clear_tokenized.append(token)
                first = False
            else:
                clear_tokenized.append('##' + token)
    return clear_tokenized

In [86]:
fixed_bpe = tokenize_bpe(example)
assert isinstance(fixed_bpe, list)
assert fixed_bpe == ['кто', 'сказал', ',', 'что', 'солдат', 'мечтае', '##т', 'стать', 'генера', '##лом', '?', 'солдат', 'мечтае', '##т', 'стать', 'хлебо', '##резо', '##м', '.']

In [87]:
fixed_bpe

['кто',
 'сказал',
 ',',
 'что',
 'солдат',
 'мечтае',
 '##т',
 'стать',
 'генера',
 '##лом',
 '?',
 'солдат',
 'мечтае',
 '##т',
 'стать',
 'хлебо',
 '##резо',
 '##м',
 '.']

In [88]:
def tokenize(text):
    reg = re.compile(r'\w+')
    return reg.findall(text.lower())

## 1. N-gram language model

A language model is a probabilistic model that calculates the probability of a sequence of tokens $P(w_1, \dots, w_T)$. Since it is difficult to estimate the joint probability directly, it is usually broken down into a product of conditional probabilities. 

$$
P(w_1, \dots, w_T) = P(w_1)\prod_{i=1}^T P(w_i \mid w_{i-1}, \dots, w_1)
$$

In practice, such conditional probabilities are difficult to estimate when the text is very long. Language models work best with a small context. To solve this problem, you can explicitly limit the length of the context by writing down the following assumption
$$
P(w_i \mid w_{i-1}, \dots, w_1) \approx P(w_i \mid w_{i-1}, \dots, w_{i-n+1}).
$$

This model is called an __n-gram language model__, as it estimates the probabilities of only n-gram tokens. Then the final probability of a sequence of tokens is written as follows

$$
P(w_1, \dots, w_T) = \prod_{i=1}^T P(w_i \mid w_{i-1}, \dots, w_{i-n+1}).
$$

Special tokens `[UNK]` can be added to the beginning of the sequence so that the condition always has a fixed-length context.

In this part the model does not need a training since it is a count-based model. That's why we are going to count the number of times we see each n-gramm. In the beginning of each sequence we will add `[UNK]`, and `[EOS]` in the end. While generating the model will generate `[EOS]`, when it is time to stop generating.

In [89]:
# from collections import defaultdict, Counter

# UNK, EOS = "[UNK]", "[EOS]"

# def count_ngrams(lines, n, tokenize=tokenize):
#     """
#     Count how many times each word occured after (n - 1) previous words
#     Input: a list of strings with space-separated tokens
#     :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

#     If the prefix is too short, it should be padded with [UNK].
#     Add [EOS] at the end of each sequence and consider it as all other token
#     """
#     counts = defaultdict(Counter)

#     # YOUR CODE
#     # YOUR CODE
#     # YOUR CODE

#     return counts

In [90]:
from collections import defaultdict, Counter

UNK, EOS = "[UNK]", "[EOS]"

def count_ngrams(lines, n, tokenize=tokenize):
    """
    Count how many times each word occured after (n - 1) previous words
    Input: a list of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    If the prefix is too short, it should be padded with [UNK].
    Add [EOS] at the end of each sequence and consider it as all other token
    """
    counts = defaultdict(Counter)

    # YOUR CODE
    # SOLUTION BELOW:
    
    for line in lines:
        tokenized = [UNK] * (n - 1) + tokenize(line) + [EOS]
        for i in range(n - 1, len(tokenized)):
            counts[tuple(tokenized[i-n+1:i])][tokenized[i]] += 1

    return counts

In [91]:
dummy_lines = aneki[-5:]
dummy_lines

['Последним раскрытым громким преступлением в Киеве было убийство Столыпина...',
 'Если бесконечное количество российских футболистов запустить на бесконечное количество футбольных полей и дать им бесконечное количество времени, то один из них когда-нибудь забьёт гол.',
 'На чемпионат мира по футболу от России нужно Юлию Самойлову отправлять, хоть какая-то надежда на победу будет.',
 'В целях профилактики от всего весной следует есть много чеснока. От женщин, кстати, тоже помогает.',
 'На моих глазах как-то две девушки затаскивали кавказца в машину. Они худенькие, а он здоровый такой, никак не хотел в машину лезть. Они попросили у меня помощи, сказали, что собаку надо в ветклинику отвезти.']

In [92]:
dummy_counts = count_ngrams(dummy_lines, n=3)

In [93]:
dummy_counts[('громким', 'преступлением')]

Counter({'в': 1})

In [94]:
dummy_counts[(UNK, UNK)]

Counter({'на': 2, 'последним': 1, 'если': 1, 'в': 1})

Now we can count the probabilities of the n-gramms.

$$ P(w_i | prefix) = \frac{Count(prefix, w_i)}{\sum_{w \in V} Count(prefix, w)} $$

In [95]:
# class NGramLanguageModel:
#     def __init__(self, corpus, n=3, tokenize=tokenize):

#         counts = count_ngrams(corpus, n, tokenize=tokenize)
#         self.n = n

#         self.probs = defaultdict(Counter)

#         # calculate the probabilities using the formula above
#         # YOUR CODE
#         # YOUR CODE
#         # YOUR CODE
        
#     def process_prefix(self, prefix):
#         if self.n == 1:
#             prefix = []
#         else:
#             prefix = prefix[-(self.n - 1):]
#             prefix = [UNK] * (self.n - 1 - len(prefix)) + prefix
            
#         return prefix

#     def get_tokens_and_probs(self, prefix):
#         prefix = self.process_prefix(prefix)

#         possible_tokens = self.probs[tuple(prefix)]

#         tokens = list(possible_tokens.keys())
#         probs = list(possible_tokens.values())

#         return tokens, probs
    
#     def get_token_prob(self, token, prefix):
#         prefix = self.process_prefix(prefix)

#         prob = self.probs[tuple(prefix)].get(token, 0)
#         return prob

In [96]:
class NGramLanguageModel:
    def __init__(self, corpus, n=3, tokenize=tokenize):

        counts = count_ngrams(corpus, n, tokenize=tokenize)
        self.n = n

        self.probs = defaultdict(Counter)

        # calculate the probabilities using the formula above
        # YOUR CODE
        # SOLUTION BELOW:
        for prefix, token_count in counts.items():
            token_sum = sum(token_count.values())
            for token, count in token_count.items():
                self.probs[prefix][token] = count / token_sum
        
    def process_prefix(self, prefix):
        if self.n == 1:
            prefix = []
        else:
            prefix = prefix[-(self.n - 1):]
            prefix = [UNK] * (self.n - 1 - len(prefix)) + prefix
            
        return prefix

    def get_tokens_and_probs(self, prefix):
        prefix = self.process_prefix(prefix)

        possible_tokens = self.probs[tuple(prefix)]

        tokens = list(possible_tokens.keys())
        probs = list(possible_tokens.values())

        return tokens, probs
    
    def get_token_prob(self, token, prefix):
        prefix = self.process_prefix(prefix)

        prob = self.probs[tuple(prefix)].get(token, 0)
        return prob

Finally, we can use the model to generate a joke

In [97]:
lm = NGramLanguageModel(aneki, n=3, tokenize=tokenize_bpe)

The generation process is always autoregressive. This means that the model's output at the previous step is fed as input to the next step. This way, the model can generate text indefinitely (or until it outputs [EOS]).

There are many techniques for choosing the next token from all possible options. For example, one can either pick the most probable token or sample a token according to the predicted probability distribution. We’ll discuss this in more detail in the further seminar. For now, we’ll use the second approach — sampling — so that we get different texts each time.

$$w_{next} \sim \frac{P(w_{next} | prefix)}{\sum_{w} P(w | prefix)}$$

In [98]:
def get_next_token(lm, prefix):
    tokens, probs = lm.get_tokens_and_probs(prefix)

    next_token = np.random.choice(tokens, p=probs)
    return next_token

In [99]:
prefix = tokenize('мужчина')

for i in range(100):
    prefix += [get_next_token(lm, prefix)]
    if prefix[-1] == EOS or len(lm.get_tokens_and_probs(prefix)[0]) == 0:
        break

print(' '.join(prefix))

мужчина должен постро ##ить дом и вот я однажды мгнове ##нно удалил это сообщение , пока еще большая редко ##сть непоня ##тливо ##е число пенсио ##неров в стране нет денег . мойша уве ##з из спальн ##и :- вы должны быть на работе - собира ##ют , счастл ##иво . [EOS]


## 2. Evaluating Language Model Quality: Perplexity

Perplexity measures how well a language model predicts the distribution of the data. It is calculated using the following formula:

$$
    {\mathbb{P}}(w_1 \dots w_T) = PPL(w_1, \dots, w_T)^{-\frac{1}{T}} = \left( \prod_i P(w_i \mid w_{i-1}, \dots, w_{i - n + 1})\right)^{-\frac{1}{T}},
$$

![](https://miro.medium.com/max/1050/1*J5kBR7XsQqRiu0p_CZEk1w.png)

You can notice that this is exactly the exponent of the cross-entropy. Therefore, the lower the perplexity, the better the model.

In [100]:
def perplexity(lm, lines, min_prob=10 ** -50., tokenize=tokenize):
    """
    :param min_prob: if P(w | ...) is smaller than min_prop, set it to min_prob.
    :returns: mean perplexity over the whole corpus
    """

    ppls = []
    for line in tqdm(lines):
        tokenized = tokenize(line)
        log_ppl = 0
        for i in range(len(tokenized)):
            log_ppl += np.log(max(
                min_prob,
                lm.get_token_prob(tokenized[i], tokenized[:i])
            ))
        ppls.append(np.exp(-log_ppl / len(tokenized)))

    return np.mean(ppls)

In [101]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

Perplexities: ppx1=81.663 ppx3=1.142 ppx10=1.103


Теперь мы можем посчитать перплексию нашей модели.

In [102]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(aneki, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(train_lines, n=n)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/31039 [00:00<?, ?it/s]

N = 1, Perplexity = 12887014247997405467423213088777838380250562560.00000


  0%|          | 0/31039 [00:00<?, ?it/s]

N = 2, Perplexity = 460881476685444201036843739570228114189619036160.00000


  0%|          | 0/31039 [00:00<?, ?it/s]

N = 3, Perplexity = 3010794325323826519754827834236485472713017655296.00000


### Probability Smoothing

Here’s the problem: every time the model encounters an n-gram in the test corpus that wasn't seen during training, it assigns it a zero probability. As a result, the entire product becomes zero, regardless of how well the rest of the text is predicted.

One way to address this is to apply probability smoothing — for example, ([Laplace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)). This technique assumes that every n-gram has been seen at least once. With a large enough corpus, this barely changes the probability distribution but helps prevent perplexity from exploding due to unseen n-grams.

$$ P(w_t \mid prefix) = \frac{Count(prefix, w_t) + \delta}{\sum_{w \in V} \big(Count(prefix, w) + \delta\big)} $$

In [103]:
class LaplaceLanguageModel(NGramLanguageModel):
    def __init__(self, corpus, n, delta=1.0, tokenize=tokenize):

        counts = count_ngrams(corpus, n, tokenize=tokenize)
        self.n = n
        self.vocab = set()
        for token_count in counts.values():
            self.vocab |= set(token_count.keys())

        self.probs = defaultdict(Counter)
        for prefix, token_count in counts.items():
            total = sum(token_count.values()) + delta * len(self.vocab)
            for token, count in token_count.items():
                self.probs[prefix][token] = (count + delta) / total

    def get_tokens_and_probs(self, prefix):
        # we want to spread some propability among all tokens
        
        tokens, probs = super().get_possible_next_tokens(prefix)
        
        left_prob = 1.0 - sum(probs)
        unseen_prob = left_prob / max(1, len(self.vocab) - len(tokens))
        
        unseen_tokens = self.vocab - set(tokens)

        return tokens + list(unseen_tokens), probs + [unseen_prob] * len(unseen_tokens)

    def get_token_prob(self, token, prefix):
        prob = super().get_token_prob(token, prefix)
        if prob > 0:
            return prob

        tokens, probs = super().get_tokens_and_probs(prefix)

        left_prob = max(1e-8, 1.0 - sum(probs))
        unseen_prob = left_prob / max(1, len(self.vocab) - len(tokens))

        return unseen_prob

In [104]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert sum(([dummy_lm.get_token_prob(w_i, ['l']) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [105]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/31039 [00:00<?, ?it/s]

N = 1, Perplexity = 39955.87579


  0%|          | 0/31039 [00:00<?, ?it/s]

N = 2, Perplexity = 30058.30273


  0%|          | 0/31039 [00:00<?, ?it/s]

N = 3, Perplexity = 70786.18871


In [106]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=1, tokenize=tokenize_bpe)
    ppx = perplexity(lm, test_lines, tokenize=tokenize_bpe)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/31039 [00:00<?, ?it/s]

N = 1, Perplexity = 2695.94599


  0%|          | 0/31039 [00:00<?, ?it/s]

N = 2, Perplexity = 3860.07277


  0%|          | 0/31039 [00:00<?, ?it/s]

N = 3, Perplexity = 15598.23201


## 4. RNNs

In [107]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
import random
import numpy as np


In [108]:
encoded = [tokenize_bpe(anek) for anek in aneki]

In [117]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

PAD, UNK = "<pad>", "<unk>"
PAD_ID, UNK_ID = 0, 1  # fixed so we can ignore PAD in loss

device: mps


In [110]:
# Build vocab from BPE strings
all_tokens = [tok for seq in encoded for tok in seq]

In [111]:
# Frequency order (keeps common tokens with low ids)
from collections import Counter
freq = Counter(all_tokens)

vocab = {PAD: PAD_ID, UNK: UNK_ID}
for i, (tok, _) in enumerate(freq.most_common(), start=2):
    vocab[tok] = i

ivocab = {i: t for t, i in vocab.items()}  # only needed for decoding demos
vocab_size = len(vocab)
print("vocab_size:", vocab_size)

vocab_size: 37580


In [112]:
def collate_raw(batch):
    return batch
    
train_loader = DataLoader(train_texts, batch_size=128, shuffle=True,
                          pin_memory=True, collate_fn=collate_raw)
test_loader  = DataLoader(test_texts,  batch_size=128, shuffle=False,
                          pin_memory=True, collate_fn=collate_raw)

In [113]:
# Batch maker: map strings –> ids, pad, and build (x,y)
def to_ids(seq, vocab=vocab, unk_id=UNK_ID):
    return torch.tensor([vocab.get(t, unk_id) for t in seq], dtype=torch.long)

@torch.no_grad()
def make_batch(batch, pad_id=PAD_ID, device=device):
    # keep sequences that have at least 2 tokens (need input and next-token target)
    seqs = [to_ids(s) for s in batch if len(s) >= 2]
    if not seqs:
        seqs = [torch.tensor([pad_id, pad_id], dtype=torch.long)]  # safety
    xs = [s[:-1] for s in seqs]  # inputs
    ys = [s[1:]  for s in seqs]  # next-token targets
    x = pad_sequence(xs, batch_first=True, padding_value=pad_id).to(device)
    y = pad_sequence(ys, batch_first=True, padding_value=pad_id).to(device)
    return x, y


In [121]:
class SimpleRNNLM(nn.Module):
    def __init__(self, vocab_size, emb=256, hidden=512, pad_idx=PAD_ID):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=pad_idx)
        self.rnn = nn.RNN(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)      # (B, T, E)
        h, _ = self.rnn(e)   # (B, T, H)
        return self.out(h)   # (B, T, V)


In [122]:
from tqdm import tqdm


def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, total_tok, total_correct = 0.0, 0, 0

    for batch in tqdm(loader, "fitting..."):
        x, y = make_batch(batch)
        logits = model(x)  # (B, T, V)
        V = logits.size(-1)
        loss = nn.functional.cross_entropy(
            logits.view(-1, V), y.view(-1), ignore_index=PAD_ID
        )
        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        with torch.no_grad():
            mask = y.ne(PAD_ID)
            pred = logits.argmax(-1)
            total_correct += (pred.eq(y) & mask).sum().item()
            tok = mask.sum().item()
            total_tok += tok
            total_loss += loss.item() * tok

    avg_loss = total_loss / max(1, total_tok)
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    acc = total_correct / max(1, total_tok)
    return avg_loss, ppl, acc


In [123]:
rnn_lm = SimpleRNNLM(vocab_size).to(device)
opt = torch.optim.AdamW(rnn_lm.parameters(), lr=3e-3)

for ep in range(1, 4):
    tr_loss, tr_ppl, tr_acc = run_epoch(rnn_lm, train_loader, optimizer=opt)
    va_loss, va_ppl, va_acc = run_epoch(rnn_lm, test_loader)
    print(f"[RNN] ep {ep:02d} | train ppl {tr_ppl:.2f} acc {tr_acc:.3f} || val ppl {va_ppl:.2f} acc {va_acc:.3f}")


fitting...:   1%|█▎                                                                                                                                                            | 7/873 [00:17<35:39,  2.47s/it]


KeyboardInterrupt: 